In [ ]:
# ==================================================
# 导入必要的库 - 使用PyTorch高级API实现RNN
# ==================================================
import torch  # PyTorch深度学习框架
from torch import nn  # PyTorch神经网络模块
from torch.nn import functional as F  # PyTorch函数式接口
from d2l import torch as d2l  # d2l工具库，提供训练、绘图等辅助功能
from RNN import load_data_time_machine  # 从自定义模块导入数据加载函数

# 本notebook与Rnn.ipynb的区别：
# - Rnn.ipynb：从零开始手动实现RNN的所有细节（教学用）
# - Rnn2.ipynb：使用PyTorch提供的nn.RNN高级API（实用高效）

In [ ]:
# ==================================================
# 设置训练参数并加载数据
# ==================================================
batch_size, num_steps = 32, 35  # 批量大小=32，每个序列的时间步数=35
# 加载时光机器文本数据集
train_iter, vocab = load_data_time_machine(batch_size, num_steps)

In [ ]:
# ==================================================
# 创建PyTorch的RNN层
# ==================================================
num_hiddens = 256  # 隐藏层单元数设为256
# 创建单层RNN：
# - 输入维度: len(vocab) - 词汇表大小（每个字符one-hot编码后的维度）
# - 隐藏层维度: num_hiddens - 隐藏状态的特征数
# nn.RNN会自动处理所有权重矩阵的初始化和前向传播计算
rnn_layer = nn.RNN(len(vocab), num_hiddens)

# 初始化隐藏状态
# 形状: (num_layers, batch_size, num_hiddens)
# - num_layers=1: 单层RNN
# - batch_size: 批量大小
# - num_hiddens: 隐藏层维度
state = torch.zeros((1, batch_size, num_hiddens))
state.shape

In [ ]:
# ==================================================
# 测试RNN层的前向传播
# ==================================================
# 创建随机输入数据，模拟one-hot编码后的序列
# 形状: (时间步数, 批量大小, 输入维度)
X = torch.rand(size=(num_steps, batch_size, len(vocab)))
# RNN前向传播
# 输入: X (num_steps, batch_size, input_size), state (num_layers, batch_size, hidden_size)
# 输出: Y (num_steps, batch_size, hidden_size), state_new (num_layers, batch_size, hidden_size)
Y, state_new = rnn_layer(X, state)
Y.shape, state_new.shape

In [ ]:
# ==================================================
# 完整的RNN模型类（封装RNN层和输出层）
# ==================================================
class RNNModel(nn.Module):
    """
    循环神经网络模型
    
    架构:
        输入 -> One-hot编码 -> RNN层 -> 全连接层 -> 输出
    
    该类将RNN层和输出层组合成一个完整的字符级语言模型
    """
    def __init__(self, rnn_layer, vocab_size, **kwargs):
        """
        初始化RNN模型
        
        参数:
            rnn_layer: PyTorch的RNN层（nn.RNN/nn.LSTM/nn.GRU）
            vocab_size: 词汇表大小
        """
        super(RNNModel, self).__init__(**kwargs)
        self.rnn = rnn_layer
        self.vocab_size = vocab_size
        self.num_hiddens = self.rnn.hidden_size  # 隐藏层维度
        
        # 判断RNN是否为双向
        # 如果RNN是双向的（之后将介绍），num_directions应该是2，否则应该是1
        if not self.rnn.bidirectional:
            self.num_directions = 1
            # 单向RNN：隐藏层到输出层的线性变换
            self.linear = nn.Linear(self.num_hiddens, self.vocab_size)
        else:
            self.num_directions = 2
            # 双向RNN：隐藏层维度翻倍（正向+反向）
            self.linear = nn.Linear(self.num_hiddens * 2, self.vocab_size)

    def forward(self, inputs, state):
        """
        前向传播
        
        参数:
            inputs: 输入序列，形状为(batch_size, num_steps)，包含字符索引
            state: 隐藏状态
        
        返回:
            output: 输出logits，形状为(num_steps*batch_size, vocab_size)
            state: 更新后的隐藏状态
        """
        # 将输入索引转换为one-hot编码
        # inputs.T: (num_steps, batch_size)
        # X: (num_steps, batch_size, vocab_size)
        X = F.one_hot(inputs.T.long(), self.vocab_size)
        X = X.to(torch.float32)
        
        # RNN前向传播
        # Y: (num_steps, batch_size, num_hiddens) - 每个时间步的隐藏状态输出
        Y, state = self.rnn(X, state)
        
        # 全连接层处理
        # 首先将Y的形状改为(时间步数*批量大小, 隐藏单元数)
        # 这样可以批量处理所有时间步的输出
        # 输出形状是(时间步数*批量大小, 词表大小)
        output = self.linear(Y.reshape((-1, Y.shape[-1])))
        return output, state

    def begin_state(self, device, batch_size=1):
        """
        初始化隐藏状态
        
        参数:
            device: 计算设备
            batch_size: 批量大小
        
        返回:
            初始隐藏状态（形状和类型取决于RNN类型）
        """
        if not isinstance(self.rnn, nn.LSTM):
            # nn.GRU和nn.RNN以张量作为隐藏状态
            # 形状: (num_directions * num_layers, batch_size, num_hiddens)
            return  torch.zeros((self.num_directions * self.rnn.num_layers,
                                 batch_size, self.num_hiddens),
                                device=device)
        else:
            # nn.LSTM以元组作为隐藏状态（包括隐藏状态h和记忆细胞c）
            return (torch.zeros((
                self.num_directions * self.rnn.num_layers,
                batch_size, self.num_hiddens), device=device),
                    torch.zeros((
                        self.num_directions * self.rnn.num_layers,
                        batch_size, self.num_hiddens), device=device))

In [ ]:
# ==================================================
# 实例化模型并测试预测功能
# ==================================================
device = d2l.try_gpu()  # 尝试使用GPU，如果不可用则使用CPU
# 创建RNN模型实例
net = RNNModel(rnn_layer, vocab_size=len(vocab))
net = net.to(device)  # 将模型移动到指定设备
# 测试预测：以"time traveller"为前缀，预测后续10个字符
# 注意：此时模型还未训练，预测结果是随机的
d2l.predict_ch8('time traveller', 10, net, vocab, device)

In [ ]:
# ==================================================
# 开始训练RNN模型
# ==================================================
num_epochs, lr = 500, 1  # 训练500轮，学习率为1
# 使用d2l提供的训练函数进行训练
# 该函数会：
# 1. 自动处理前向传播、损失计算、反向传播和参数更新
# 2. 每10轮打印一次预测结果
# 3. 实时绘制困惑度变化曲线
# 4. 应用梯度裁剪防止梯度爆炸
d2l.train_ch8(net, train_iter, vocab, lr, num_epochs, device)